In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import sys

In [2]:
current_path = os.path.dirname(os.path.realpath(__file__)) if '__file__' in locals() else os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(current_path, '..'))
SRC_DIR = os.path.join(ROOT_DIR,'src')
DATA_PATH = os.path.join(ROOT_DIR, "data", "raw")
sys.path.append(SRC_DIR)

In [3]:
from preprocessing import build_dataset
from modeling import *

In [4]:
df = build_dataset(data_path = DATA_PATH)

print(df.shape)
df.head()

(117604, 45)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,is_approved,...,payment_installments,payment_value,product_volume,order_month,order_dayofweek,estimated_cost,total_cost,profit,profit_margin,target
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,1,...,1.0,18.12,1976.0,10,0,10.0,18.72,11.27,0.602030,1
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,1,...,1.0,2.00,1976.0,10,0,10.0,18.72,11.27,0.602030,1
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,1,...,1.0,18.59,1976.0,10,0,10.0,18.72,11.27,0.602030,1
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,1,...,1.0,141.46,4693.0,7,1,8.0,30.76,87.94,2.858908,1
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,1,...,3.0,179.12,9576.0,8,2,8.4,27.62,132.28,4.789283,1


In [5]:
target = "target"

features = [
    # числовые
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_photos_qty",

    # категории (уже закодированы в EDA)
    "customer_state",
    "seller_state",
    "product_category_name"
]

In [6]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2804
)

print(X_train.shape, X_test.shape)

(94083, 7) (23521, 7)


In [7]:
X = df[features]

print(X.isna().sum())

product_length_cm        0
product_height_cm        0
product_width_cm         0
product_photos_qty       0
customer_state           0
seller_state             0
product_category_name    0
dtype: int64


In [8]:
models = get_models()

results = []

for name, model in models.items():

    pipe = build_pipeline(name, model)

    pipe = train_model(pipe, X_train, y_train)

    score = evaluate_model(pipe, X_test, y_test)

    results.append({
        "Model": name,
        "ROC-AUC": score,
        "Scaled": name in ["logreg", "knn", "svm"]
    })

    print(f"{name}: {score:.4f}")

logreg: 0.6650
random_forest: 0.9523
gradient_boosting: 0.7905
knn: 0.8861
svm: 0.7771


In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

results_df

,Model,ROC-AUC,Scaled
0,random_forest,0.952282,False
1,knn,0.886150,True
2,gradient_boosting,0.790451,False
3,svm,0.777074,True
4,logreg,0.664970,True


In [10]:
rf_grid = tune_random_forest(X_train, y_train)

print("Best params:", rf_grid.best_params_)
print("Best CV score:", rf_grid.best_score_)

Best params: {'model__max_depth': None, 'model__n_estimators': 200}
Best CV score: 0.941451787988172


In [ ]:
svm_grid = tune_svm(X_train, y_train)

print("Best params:", svm_grid.best_params_)
print("Best CV score:", svm_grid.best_score_)

In [ ]:
results.append({
    "Model": "random_forest_tuned",
    "ROC-AUC": rf_grid.best_score_,
    "Scaled": False
})

results.append({
    "Model": "svm_tuned",
    "ROC-AUC": svm_grid.best_score_,
    "Scaled": True
})

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)

results_df

In [ ]:
best_model = rf_grid.best_estimator_

final_score = evaluate_model(best_model, X_test, y_test)

print("Final ROC-AUC:", final_score)

In [ ]:
save_model(best_model)